# Kredi Notu Sınıflandırma - Şampiyon Hibrit Model (Stacking Ensemble)

Bu notebook, projenin tüm aşamalarından geçerek belirlenen **en iyi performanslı (Champion)** model mimarisini içermektedir. Gereksiz ara adımlardan arındırılmış, doğrudan sonuç odaklı nihai üretim kodudur.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import os

# Görselleştirme ayarları
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

print("✅ Kütüphaneler yüklendi.")

## 1. Veri Hazırlığı ve Özellik Seçimi
Modelin en yüksek başarıya ulaştığı dürüst veri setini yüklüyoruz (ID ve İsim bilgileri dışlanmıştır).

In [ ]:
# Veri Yolunu Belirleyin
data_path = "1_Veri_Manipülasyonu/Ciktilar/12_Kategorik_Kodlama/ml_hazir_sayisal_veri.xlsx"
bridge_path = "3_Siniflandirma_Modelleri/Modeller/selected_features.joblib"

if not os.path.exists(data_path):
    print("❌ HATA: Veri dosyası bulunamadı. Lütfen yolu kontrol edin.")
else:
    df = pd.read_excel(data_path)
    df = df[df["Kredi Skoru"].notna()] # Sadece etiketli veri
    
    # Seçilmiş özellik köprüsünü yükle (Veya manuel liste kullan)
    if os.path.exists(bridge_path):
        selected_features = joblib.load(bridge_path)
    else:
        # Köprü bulunamazsa en önemli 20 özelliği manuel tanımlıyoruz
        selected_features = ['Gecikme_Gun_Sayisi', 'Kredi_Karisimi', 'Odeme_Davranisi', 
                             'Kredi_Gecmisi_Yasi', 'Yillik_Gelir', 'Mevcut_Borc', 
                             'Kredi_Sorgu_Sayisi', 'Kredi_Karti_Sayisi', 'Faiz_Orani']
    
    X = df[selected_features]
    y = df["Kredi Skoru"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"✅ Veri Hazır: {X_train.shape[0]} eğitim örneği, {X_test.shape[0]} test örneği.")

## 2. Şampiyon Hibrit Model (Stacking Ensemble)
Üç güçlü algoritmayı taban model (Level 0) olarak kullanıp, kararı Lojistik Regresyon (Level 1) ile optimize ediyoruz.

In [ ]:
print("🚀 Hibrit model eğitiliyor (Bu işlem 2-3 dakika sürebilir)...")

# Taban Modeller
level0 = [
    ('et', ExtraTreesClassifier(n_estimators=100, max_depth=20, n_jobs=-1)),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1)),
    ('hgb', HistGradientBoostingClassifier(max_iter=100))
]

# Meta Model (Karar Verici)
level1 = LogisticRegression()

champion_model = StackingClassifier(estimators=level0, final_estimator=level1, cv=5)
champion_model.fit(X_train, y_train)

print("✅ Eğitim Tamamlandı!")

## 3. Sonuçların Değerlendirilmesi

In [ ]:
y_pred = champion_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\n🔥 NİHAİ TEST DOĞRULUĞU: %{acc*100:.2f}")
print("-"*30)
print(classification_report(y_test, y_pred, target_names=['İyi (0)', 'Standart (1)', 'Kötü (2)']))

# Karmaşıklık Matrisi
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['İyi', 'Standart', 'Kötü'], 
            yticklabels=['İyi', 'Standart', 'Kötü'])
plt.title("Şampiyon Model: Karmaşıklık Matrisi")
plt.ylabel('Gerçek Değerler')
plt.xlabel('Tahmin Edilenler')
plt.show()